# 01 — Data Processing: Multi-Source Download (Yahoo Finance / Stooq / AlphaVantage)

**Project:** Reproducing Fischer & Krauss (2018) — LSTM Networks for Financial Market Predictions
**Student:** Rahul Choudhary (MA25C033)

## Purpose of this notebook
Download daily OHLCV price data for the S&P 500 universe using **three data
sources with automatic fallback**:

1. **Yahoo Finance** (via `yfinance`) — primary source, free, no API key.
2. **Stooq** — fallback if Yahoo Finance fails for a ticker (also free, no key).
3. **Alpha Vantage** — final fallback (requires a free API key, but has a strict
   rate limit of 5 requests/minute on the free tier).

Using three sources adds robustness: a ticker that is temporarily rate-limited
or missing on one source can often still be retrieved from another.

## Known limitation (documented, per Prof. Neelesh's guidance)
This uses **today's S&P 500 constituent list** applied across the requested
historical date range. This introduces **survivorship bias**: companies that
were removed from the index in the past (bankruptcy, acquisition, delisting)
are not included, unlike the original paper's survivor-bias-free Thomson
Reuters reconstruction. This must be restated in the final report.

## Output
One CSV file per ticker saved to `../data/raw/`, each containing daily
Open/High/Low/Close/Volume, plus a `_download_summary.csv` logging which
source succeeded for each ticker (or if all three failed).


## 1. Setup: install and import dependencies

In [ ]:
# Uncomment to install (run once)
# %pip install yfinance pandas requests --quiet

import time
import io
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
import yfinance as yf

print("Libraries loaded OK.")


## 2. Configuration

Set the date range, output folder, and (optionally) your Alpha Vantage API
key below. Get a free key at https://www.alphavantage.co/support/#api-key
if you want the third fallback source active — otherwise leave it as
`None` and the pipeline will just use Yahoo Finance + Stooq.


In [ ]:
START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Optional: paste your free Alpha Vantage key here, or leave as None to skip it
ALPHAVANTAGE_API_KEY = None  # e.g. "YOUR_KEY_HERE"

# Set to an integer to only download the first N tickers (useful for a quick
# test run before committing to the full S&P 500 universe).
TICKER_LIMIT = 20

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Output directory: {RAW_DATA_DIR.resolve()}")


## 3. Get the S&P 500 ticker list

Scraped from the public Wikipedia constituent list. This is **today's**
list — see the survivorship-bias note above.


In [ ]:
def get_sp500_tickers() -> list[str]:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = pd.read_html(url)
    sp500_table = tables[0]
    tickers = sp500_table["Symbol"].str.replace(".", "-", regex=False).tolist()
    return tickers


tickers = get_sp500_tickers()
if TICKER_LIMIT:
    tickers = tickers[:TICKER_LIMIT]

print(f"Using {len(tickers)} tickers.")
print(tickers[:10], "...")


## 4. Data source functions

Each function takes a ticker and date range, and returns a standardized
DataFrame with columns `[date, open, high, low, close, volume]`, or `None`
if the download failed.


### 4.1 Yahoo Finance (primary)

In [ ]:
def download_yahoo(ticker: str, start: str, end: str) -> pd.DataFrame | None:
    try:
        df = yf.download(ticker, start=start, end=end, auto_adjust=True,
                          progress=False)
        if df.empty:
            return None
        df = df.reset_index()
        df.columns = [c.lower() if isinstance(c, str) else c[0].lower()
                      for c in df.columns]
        df = df.rename(columns={"date": "date"})
        return df[["date", "open", "high", "low", "close", "volume"]]
    except Exception as e:
        print(f"  [Yahoo] {ticker} failed: {e}")
        return None


### 4.2 Stooq (fallback 1)

In [ ]:
def download_stooq(ticker: str, start: str, end: str) -> pd.DataFrame | None:
    # Stooq uses a simple CSV download URL, US tickers use the ".us" suffix.
    stooq_symbol = f"{ticker.lower()}.us"
    url = (
        f"https://stooq.com/q/d/l/?s={stooq_symbol}&d1={start.replace('-', '')}"
        f"&d2={end.replace('-', '')}&i=d"
    )
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        if "Date" not in resp.text:
            return None  # Stooq returns a plain error message body on failure
        df = pd.read_csv(io.StringIO(resp.text))
        df.columns = [c.lower() for c in df.columns]
        df["date"] = pd.to_datetime(df["date"])
        return df[["date", "open", "high", "low", "close", "volume"]]
    except Exception as e:
        print(f"  [Stooq] {ticker} failed: {e}")
        return None


### 4.3 Alpha Vantage (fallback 2, requires API key)

In [ ]:
def download_alphavantage(ticker: str, api_key: str | None) -> pd.DataFrame | None:
    if not api_key:
        return None
    url = (
        "https://www.alphavantage.co/query"
        f"?function=TIME_SERIES_DAILY&symbol={ticker}&outputsize=full"
        f"&apikey={api_key}&datatype=csv"
    )
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text))
        if "timestamp" not in df.columns:
            return None  # error payload (e.g. rate limit hit)
        df = df.rename(columns={"timestamp": "date"})
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date")
        return df[["date", "open", "high", "low", "close", "volume"]]
    except Exception as e:
        print(f"  [AlphaVantage] {ticker} failed: {e}")
        return None


## 5. Master download function with fallback logic

Tries Yahoo Finance first, then Stooq, then Alpha Vantage (if a key is
set). Returns both the data and which source succeeded, for logging.


In [ ]:
def download_with_fallback(ticker: str, start: str, end: str,
                            av_key: str | None = None) -> tuple[pd.DataFrame | None, str]:
    df = download_yahoo(ticker, start, end)
    if df is not None and len(df) > 0:
        return df, "yahoo"

    df = download_stooq(ticker, start, end)
    if df is not None and len(df) > 0:
        return df, "stooq"

    df = download_alphavantage(ticker, av_key)
    if df is not None and len(df) > 0:
        return df, "alphavantage"

    return None, "failed"


## 6. Run the download loop

This downloads every ticker in `tickers`, saves one CSV per ticker to
`data/raw/`, and logs which source was used (or if it failed entirely).

**Note:** Alpha Vantage free tier allows only 5 requests/minute — if you
enabled it above, this loop pauses briefly between AlphaVantage calls to
respect that limit.


In [ ]:
download_log = []

for i, ticker in enumerate(tickers):
    print(f"[{i+1}/{len(tickers)}] {ticker}...", end=" ")
    df, source = download_with_fallback(ticker, START_DATE, END_DATE,
                                         ALPHAVANTAGE_API_KEY)

    if df is not None:
        df.to_csv(RAW_DATA_DIR / f"{ticker}.csv", index=False)
        print(f"OK ({source}, {len(df)} rows)")
    else:
        print("FAILED on all sources")

    download_log.append({"ticker": ticker, "source": source,
                          "rows": len(df) if df is not None else 0})

    # Be polite to free APIs / avoid rate limits
    if source == "alphavantage":
        time.sleep(13)  # ~5 requests/minute limit
    else:
        time.sleep(0.5)

log_df = pd.DataFrame(download_log)
log_df.to_csv(RAW_DATA_DIR / "_download_summary.csv", index=False)
print("\nDownload complete. Summary saved to _download_summary.csv")


## 7. Summary report

In [ ]:
print("Download source breakdown:")
print(log_df["source"].value_counts())
print()

failed = log_df[log_df["source"] == "failed"]
if len(failed) > 0:
    print(f"{len(failed)} tickers failed on ALL sources:")
    print(failed["ticker"].tolist())
else:
    print("All tickers downloaded successfully.")


## 8. Sanity check — plot one ticker

Quick visual check that the downloaded data looks reasonable before moving
on to `02_baseline_logistic_regression.ipynb`.


In [ ]:
import matplotlib.pyplot as plt

sample_ticker = log_df[log_df["source"] != "failed"]["ticker"].iloc[0]
sample_df = pd.read_csv(RAW_DATA_DIR / f"{sample_ticker}.csv",
                         parse_dates=["date"])

plt.figure(figsize=(10, 4))
plt.plot(sample_df["date"], sample_df["close"])
plt.title(f"{sample_ticker} — Close Price ({START_DATE} to {END_DATE})")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.tight_layout()
plt.show()

print(sample_df.head())
print(f"\n{sample_ticker}: {len(sample_df)} trading days downloaded.")


## Next step

Proceed to **`02_baseline_logistic_regression.ipynb`**, which loads the
CSVs saved in `data/raw/`, builds the feature/target tables (per
Fischer & Krauss Section 3.2), and trains the Logistic Regression baseline.
